# Create the NBA stats Google Sheet in your Google account

Run **Runtime → Run all**. Google will ask you to sign in once.

That creates a spreadsheet **you own**, with:
- **One tab per season** (`2021-22` … `2025-26`) — 500 players, 423 stats
- **All_seasons** — 2,500 rows for Pivot Tables and Explore
- **Dashboard** — scoring/VORP leaders plus starter charts

When it finishes, the live Google Sheets URL is printed at the bottom.

In [ ]:
from google.colab import auth

auth.authenticate_user()
print("Signed in.")

In [ ]:
import urllib.request
from pathlib import Path

import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

XLSX = Path("/tmp/nba_player_stats_last_five_years.xlsx")
URLS = [
    "https://raw.githubusercontent.com/ashwinkren/ashwinkren/cursor/nba-player-stats-e7f6/data/nba_player_stats_last_five_years.xlsx",
    "https://cdn.jsdelivr.net/gh/ashwinkren/ashwinkren@cursor/nba-player-stats-e7f6/data/nba_player_stats_last_five_years.xlsx",
]

last_err = None
for url in URLS:
    try:
        print("Downloading", url)
        urllib.request.urlretrieve(url, XLSX)
        break
    except Exception as exc:
        last_err = exc
        print("  failed:", exc)
else:
    raise RuntimeError(f"Could not download workbook: {last_err}")

print("Local file", XLSX, "size", XLSX.stat().st_size, "bytes")

creds, _ = google.auth.default()
drive = build("drive", "v3", credentials=creds)

media = MediaFileUpload(
    XLSX,
    mimetype="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    resumable=True,
)
created = (
    drive.files()
    .create(
        body={
            "name": "NBA player stats — last five years",
            "mimeType": "application/vnd.google-apps.spreadsheet",
        },
        media_body=media,
        fields="id,name,webViewLink",
        supportsAllDrives=True,
    )
    .execute()
)

file_id = created["id"]
sheet_url = created["webViewLink"]

drive.permissions().create(
    fileId=file_id,
    body={"type": "anyone", "role": "reader"},
    fields="id",
).execute()

print("\nGoogle Sheet is ready (you are the owner):")
print(sheet_url)